# Dynamics

Plots opinion trajectories, convergence, and equilibrium opinion shift as a function of the fraction of AI adopters from Friedkin-Johnsen simulations (Section 3.2 and Appendix D). Reads from `outputs/dynamics/`.

In [1]:
import os
os.chdir("../")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import struct
import glob
from src import utils

def load_trajectory(filepath):
    with open(filepath, 'rb') as f:
        n_agents = struct.unpack('i', f.read(4))[0]
        n_timesteps = struct.unpack('i', f.read(4))[0]
        data = np.frombuffer(f.read(), dtype=np.float32)
        data = data.reshape((n_agents, n_timesteps), order='F')
    return data

sns.set_theme(context='paper', style='ticks', font_scale=1)

os.environ['PATH'] = f"{os.path.expanduser('~/.TinyTeX/bin/x86_64-linux')}:{os.environ['PATH']}"

In [ ]:
name = "dynamics"
exp_name = "dynamics"
width_pt = 469

network = "twitter"
# dataset = "ukp"
dataset = "semeval"
# task = "writing"
task = "improvement"
topic = "abortion"
model_name = "google/gemma-3-12b-it"
quantification_method = "centroid"

stubbornness_means = [0.1, 0.3, 0.5, 0.7, 0.9]
transformed_pcts = [0, 0.2, 0.4, 0.6, 0.8, 1.0]

line_plot_community_ratio = 0.4
line_plot_stubbornness_mean = 0.3
line_plot_transformed_pct = 0.6

model_str = model_name.replace("/", "_")
results_dir = "outputs/dynamics"


def result_path(stubbornness_mean, transformed_pct):
    return (
        f"{results_dir}/{exp_name}__network={network}"
        f"__stubbornness_mean={stubbornness_mean}"
        f"__transformed_pct={transformed_pct}"
        f"__dataset={dataset}__task={task}__topic={topic}"
        f"__model={model_str}__quantification_method={quantification_method}.json"
    )


def traj_path(stubbornness_mean, transformed_pct, seed):
    base = result_path(stubbornness_mean, transformed_pct)[:-len(".json")]
    return f"{base}_seed{seed}_trajectory.bin"

## Average opinion trajectory over time vs. AI adoption

In [ ]:
records = []
for pct in transformed_pcts:
    with open(result_path(line_plot_stubbornness_mean, pct)) as f:
        data = json.load(f)
    for seed_idx, result in enumerate(data["seed_results"]):
        for t, stats in enumerate(result["internal_trajectory"]):
            records.append({
                "timestep": t,
                "seed": seed_idx,
                "opinion": stats["overall_mean"],
                "transformed_pct": pct,
            })
trajectory_df = pd.DataFrame(records)

trajectory_df = trajectory_df[trajectory_df["timestep"] < trajectory_df["timestep"].max() - 50]

utils.latexify()
fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

palette = sns.color_palette("crest", n_colors=len(transformed_pcts))
sns.lineplot(
    data=trajectory_df,
    x="timestep",
    y="opinion",
    hue="transformed_pct",
    palette=palette,
    errorbar=("ci", 95),
    ax=ax,
)
sns.despine(ax=ax)
ax.set_xlabel("Timestep")
ax.set_ylabel("Average opinion")

handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{float(l) * 100:.0f}\\%" for l in labels]
ax.legend(
    handles[::-1],
    new_labels[::-1],
    title="AI adoption",
    loc="upper right",
    framealpha=0.9,
)

fig.tight_layout()
fig.savefig(
    f"figures/{name}__trajectory__{model_str}__network={network}__dataset={dataset}__topic={topic}"
    f"__cr={line_plot_community_ratio}__sm={line_plot_stubbornness_mean}.pdf",
    dpi=300,
)
plt.close()

## Convergence: max opinion change per timestep

In [ ]:
dynamics_pattern = (
    f"{results_dir}/{exp_name}__network=*"
    f"__stubbornness_mean=*"
    f"__transformed_pct={line_plot_transformed_pct}"
    f"__dataset={dataset}__task={task}__topic=*"
    f"__model={model_str}__quantification_method={quantification_method}.json"
)
network_topic_pairs = set()
for path in glob.glob(dynamics_pattern):
    basename = os.path.basename(path).replace(".json", "")
    parts = dict(p.split("=", 1) for p in basename.split("__")[1:])
    network_topic_pairs.add((parts["network"], parts["topic"]))
network_topic_pairs = sorted(network_topic_pairs)
print(f"Found {len(network_topic_pairs)} (network, topic) pairs: {network_topic_pairs}")

utils.latexify()

for nw, tp in network_topic_pairs:
    records = []
    for sm in stubbornness_means:
        path = (
            f"{results_dir}/{exp_name}__network={nw}"
            f"__stubbornness_mean={sm}"
            f"__transformed_pct={line_plot_transformed_pct}"
            f"__dataset={dataset}__task={task}__topic={tp}"
            f"__model={model_str}__quantification_method={quantification_method}.json"
        )
        try:
            with open(path) as fp:
                data = json.load(fp)
        except FileNotFoundError:
            continue
        for seed_idx, result in enumerate(data["seed_results"]):
            for t, mc in enumerate(result["max_change_per_step"]):
                records.append({
                    "timestep": t,
                    "seed": seed_idx,
                    "max_change": mc,
                    "stubbornness_mean": sm,
                })
    if not records:
        continue
    convergence_df = pd.DataFrame(records)

    fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    palette = sns.color_palette("crest", n_colors=len(stubbornness_means))
    sns.lineplot(
        data=convergence_df,
        x="timestep",
        y="max_change",
        hue="stubbornness_mean",
        palette=palette,
        errorbar=("ci", 95),
        ax=ax,
    )
    sns.despine(ax=ax)
    ax.set_xlabel("Timestep")
    ax.set_ylabel(r"Maximum opinion change")

    handles, labels = ax.get_legend_handles_labels()
    new_labels = [f"{float(l):.1f}" for l in labels]
    ax.legend(
        handles[::-1],
        new_labels[::-1],
        title="Average stubbornness",
        loc="upper right",
        framealpha=0.9,
    )

    fig.tight_layout()
    fig.savefig(
        f"figures/{name}__convergence_by_sm__{model_str}__network={nw}__dataset={dataset}__task={task}__topic={tp}"
        f"__cr={line_plot_community_ratio}__pct={line_plot_transformed_pct}.pdf",
        dpi=300,
    )
    plt.close()

Found 9 (network, topic) pairs: [('facebook', 'abortion'), ('facebook', 'atheism'), ('facebook', 'feminism'), ('gplus', 'abortion'), ('gplus', 'atheism'), ('gplus', 'feminism'), ('twitter', 'abortion'), ('twitter', 'atheism'), ('twitter', 'feminism')]


## Convergence: change in average opinion per timestep

In [ ]:
utils.latexify()

for nw, tp in network_topic_pairs:
    records = []
    for sm in stubbornness_means:
        path = (
            f"{results_dir}/{exp_name}__network={nw}"
            f"__stubbornness_mean={sm}"
            f"__transformed_pct={line_plot_transformed_pct}"
            f"__dataset={dataset}__task={task}__topic={tp}"
            f"__model={model_str}__quantification_method={quantification_method}.json"
        )
        try:
            with open(path) as fp:
                data = json.load(fp)
        except FileNotFoundError:
            continue
        for seed_idx, result in enumerate(data["seed_results"]):
            for t, mc in enumerate(result["mean_change_per_step"]):
                records.append({
                    "timestep": t,
                    "seed": seed_idx,
                    "mean_change": mc,
                    "stubbornness_mean": sm,
                })
    if not records:
        continue
    mean_convergence_df = pd.DataFrame(records)

    fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    palette = sns.color_palette("crest", n_colors=len(stubbornness_means))
    sns.lineplot(
        data=mean_convergence_df,
        x="timestep",
        y="mean_change",
        hue="stubbornness_mean",
        palette=palette,
        errorbar=("ci", 95),
        ax=ax,
    )
    sns.despine(ax=ax)
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Change in average opinion")

    handles, labels = ax.get_legend_handles_labels()
    new_labels = [f"{float(l):.1f}" for l in labels]
    ax.legend(
        handles[::-1],
        new_labels[::-1],
        title="Average stubbornness",
        loc="upper right",
        framealpha=0.9,
    )

    fig.tight_layout()
    fig.savefig(
        f"figures/{name}__mean_convergence_by_sm__{model_str}__network={nw}__dataset={dataset}__task={task}__topic={tp}"
        f"__cr={line_plot_community_ratio}__pct={line_plot_transformed_pct}.pdf",
        dpi=300,
    )
    plt.close()